# M06A — Non-parametric Classification: Parzen & KNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tulip-lab/pattern-classification-lab/blob/develop/M06-Nonparametric-Methods/M06A-Nonparametric-Classification-Parzen-and-KNN.ipynb)

**Pattern Classification practical · approximately 2 hours · M06 · CLO2, CLO3**

This is the canonical student-facing practical for session `M06A`.
It was newly authored for the Pattern Classification Lab using concepts and
public migration references from `tulip-lab/flip01`: `F1A-04-ParzenWindow.ipynb`, `F1A-05-KNN.ipynb`. No legacy
notebook cells or outputs are copied verbatim. The implementation uses current
public APIs and synthetic or scikit-learn-bundled data.

**Licence:** code cells are available under MPL-2.0; original narrative teaching
content is available under CC BY-NC-SA 4.0. Third-party works and datasets retain
their own terms. See the repository `LICENSE`, `CONTENT-LICENSE.md`, and
`NOTICE.md`.

## Overview

Parzen windows and KNN defer much of their work until prediction time. You will compare their decision boundaries on nonlinear data and investigate how local smoothing controls generalisation.

Work through the guided cells first, then complete the challenge in your own
copy. Keep your interpretation beside the code: a result without an argument
is not complete evidence.

## Learning Objectives

- Fit a class-conditional Parzen-window classifier using kernel density estimation.
- Fit and interpret a K-nearest-neighbours classifier.
- Relate bandwidth and neighbour count to bias, variance, and boundary smoothness.
- Compare non-parametric models under a shared evaluation protocol.

## Expected Output and Evidence

Retain the following evidence in your executed notebook:

- fitted Parzen and KNN classifiers under a shared split
- boundary figures paired with numeric held-out metrics
- a cross-validated smoothing choice and bias/variance interpretation

Do not include credentials, private data, or another learner's work.

## Prerequisites

- Distances and neighbourhoods
- Conditional probability
- Train/test splitting and scaling

## Session Plan

| Time | Activity | Evidence |
|---:|---|---|
| 0–10 min | Orientation and diagnostic prompt | Initial prediction or sketch |
| 10–30 min | Background concepts and setup | Concept notes and reproducible environment |
| 30–75 min | Guided practice | Executed analysis with interpretations |
| 75–105 min | Student challenge | Independent model choice and evidence |
| 105–115 min | Testing and debugging | Passing checks and resolved issues |
| 115–120 min | Submission and reflection | Concise evidence-based reflection |

## Background Concepts

A Parzen estimator places a kernel around every training point. The bandwidth $h$ controls smoothing: small $h$ produces a variable estimate; large $h$ can erase structure. A class-conditional density plus a class prior gives a Bayesian classifier.

KNN predicts from the labels of nearby training observations. Small $k$ is flexible and noise-sensitive; large $k$ is smoother and can underfit. Because both methods depend on distance, scaling is part of the model and must be learnt from training data only.

## Accessibility and Responsible Use

Every visual result in the guided path is paired with a printed numeric summary. Do not rely on colour alone in your challenge evidence: add labels, line styles, markers, or a compact table as appropriate. Add concise alt text when exporting figures for another format. Use only public, approved, or synthetic data.

Follow the collaboration, acknowledgement, and AI-use rules published by your current offering; this institution-neutral practical does not define those rules.

## Setup

Run the next cell before starting. All datasets are bundled with scikit-learn or generated locally; no download is required.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.datasets import make_moons
from sklearn.neighbors import KernelDensity, KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

## Guided Practice

Follow the sequence and pause at each interpretation prompt before moving on.

### 1. Generate nonlinear data

The two moons make a linear boundary inappropriate but remain easy to visualise.

In [ ]:
X, y = make_moons(n_samples=650, noise=0.24, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=18, alpha=0.65)
plt.title("Training data")
plt.show()

### 2. Wrap class-conditional kernel densities as a classifier

The class prior is added in log space to each density score.

In [ ]:
class ParzenClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, bandwidth=0.35):
        self.bandwidth = bandwidth

    def fit(self, X, y):
        self.classes_, counts = np.unique(y, return_counts=True)
        self.log_priors_ = np.log(counts / counts.sum())
        self.models_ = [KernelDensity(bandwidth=self.bandwidth).fit(X[y == cls]) for cls in self.classes_]
        return self

    def decision_function(self, X):
        return np.column_stack([
            model.score_samples(X) + log_prior
            for model, log_prior in zip(self.models_, self.log_priors_)
        ])

    def predict(self, X):
        return self.classes_[self.decision_function(X).argmax(axis=1)]

parzen = Pipeline([
    ("scale", StandardScaler()),
    ("parzen", ParzenClassifier(bandwidth=0.35)),
])
parzen.fit(X_train, y_train)
parzen_accuracy = accuracy_score(y_test, parzen.predict(X_test))
print(f"Parzen test accuracy: {parzen_accuracy:.3f}")

### 3. Fit KNN with the same preprocessing

Using the same split and scaling makes the comparison interpretable.

In [ ]:
knn = Pipeline([
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=15)),
])
knn.fit(X_train, y_train)
knn_accuracy = accuracy_score(y_test, knn.predict(X_test))
print(f"KNN test accuracy: {knn_accuracy:.3f}")

### 4. Compare the boundaries

Look for isolated regions, excessive wiggles, and smoothing across genuine class structure.

In [ ]:
def plot_boundary(ax, estimator, title):
    x0 = np.linspace(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4, 220)
    x1 = np.linspace(X[:, 1].min() - 0.4, X[:, 1].max() + 0.4, 220)
    xx, yy = np.meshgrid(x0, x1)
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    zz = estimator.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", s=18, edgecolor="k", linewidth=0.2)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_boundary(axes[0], parzen, f"Parzen (accuracy={parzen_accuracy:.2f})")
plot_boundary(axes[1], knn, f"KNN (accuracy={knn_accuracy:.2f})")
plt.tight_layout()
plt.show()

## Student Challenge

Use five-fold stratified cross-validation on the training set to compare Parzen bandwidths `[0.1, 0.2, 0.35, 0.6, 1.0]` and KNN neighbour counts `[1, 5, 15, 35, 75]`. Select one model family and setting, evaluate it once on the test set, and use boundary shape to support your explanation.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
challenge_results = []

# TODO: evaluate each Parzen bandwidth and KNN neighbour count using `cv`.
# Store model name, setting, mean accuracy, and standard deviation.
challenge_results

### Challenge success criteria

- All settings use identical folds and training data.
- Scaling stays inside each cross-validation pipeline.
- Selection uses cross-validation rather than test accuracy.
- The explanation links smoothing to boundary shape and uncertainty.

## Testing and Debugging

Run these checks after the guided practice. If one fails, inspect shapes, label order, random seeds, and whether preprocessing was fitted only on training data.

In [ ]:
# Exercise missing-information behaviour for a distance-based model.
try:
    knn.predict(np.array([[np.nan, 0.0]]))
except ValueError as error:
    print("Expected missing-value failure:", str(error).split("\n")[0])
else:
    raise AssertionError("KNN should reject NaN input in this workflow.")

single_prediction = knn.predict(X_test[:1])
assert single_prediction.shape == (1,)
print("Edge check: a single observation returns one class label.")

In [ ]:
assert set(np.unique(parzen.predict(X_test))).issubset({0, 1})
assert set(np.unique(knn.predict(X_test))).issubset({0, 1})
assert parzen_accuracy > 0.75 and knn_accuracy > 0.75
assert parzen.named_steps["parzen"].bandwidth > 0
print("M06A guided-practice checks passed.")

## Extension Task

Add 20 irrelevant Gaussian features to the moon data. Repeat the comparison and explain the effect using the curse of dimensionality.

## Submission and Reflection

Submit your completed notebook (or an HTML export) with outputs visible. Include: 

1. the challenge code and result;
2. one figure or compact results table;
3. a 150–250 word reflection answering: **What changed, why did it change, and what evidence supports your explanation?**

Follow your offering's published collaboration, acknowledgement, and AI-use rules. You remain responsible for checking and explaining all submitted work.

## References

- Sources: `F1A-04-ParzenWindow.ipynb` and `F1A-05-KNN.ipynb`.
- scikit-learn User Guide: [Density estimation](https://scikit-learn.org/stable/modules/density.html)
- scikit-learn User Guide: [Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)